In [0]:
# ============================================================
# Import Required Libraries
# ============================================================

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("main_category", StringType()),
    StructField("title", StringType()),
    StructField("average_rating", DoubleType()),
    StructField("rating_number", LongType()),
    StructField("features", ArrayType(StringType())),
    StructField("description", ArrayType(StringType())),
    StructField("price", DoubleType()),
    StructField("images", ArrayType(MapType(StringType(), StringType()))),
    StructField("videos", ArrayType(MapType(StringType(), StringType()))),
    StructField("store", StringType()),
    StructField("categories", ArrayType(StringType())),
    StructField("details", MapType(StringType(), StringType())),
    StructField("parent_asin", StringType()),
    StructField("bought_together", StringType())
])

meta_df = spark.read.schema(schema).json(
    "/Volumes/project/default/data/meta_Appliances.jsonl"
)

In [0]:
from pyspark.sql.types import *

review_schema = StructType([
    StructField("asin", StringType()),
    StructField("helpful_vote", LongType()),

    StructField(
        "images",
        ArrayType(
            StructType([
                StructField("attachment_type", StringType()),
                StructField("large_image_url", StringType()),
                StructField("medium_image_url", StringType()),
                StructField("small_image_url", StringType())
            ])
        )
    ),

    StructField("parent_asin", StringType()),
    StructField("rating", DoubleType()),
    StructField("text", StringType()),
    StructField("timestamp", LongType()),
    StructField("title", StringType()),
    StructField("user_id", StringType()),
    StructField("verified_purchase", BooleanType())
])

reviews_df = spark.read.schema(review_schema).json(
    "/Volumes/project/default/data/Appliances.jsonl"
)

In [0]:
# ============================================================
# Dataset Dimensions
# ============================================================

summary = [
    ("Reviews", reviews_df.count(), len(reviews_df.columns)),
    ("Metadata", meta_df.count(), len(meta_df.columns))
]

summary_df = spark.createDataFrame(
    summary,
    ["Dataset", "Rows", "Columns"]
)

display(summary_df)

Dataset,Rows,Columns
Reviews,2128605,10
Metadata,94327,14


In [0]:
print("Reviews Dataset")
display(reviews_df.limit(5))

print("Metadata Dataset")
display(meta_df.limit(5))

Reviews Dataset


asin,helpful_vote,images,parent_asin,rating,text,timestamp,title,user_id,verified_purchase
B01N0TQ0OH,0,List(),B01N0TQ0OH,5.0,work great. use a new one every month,1519317108692,Work great,AGKHLEW2SOWHNMFQIJGBECAF7INQ,true
B07DD2DMXB,0,List(),B07DD37QPZ,5.0,Little on the thin side,1664746863446,excellent product,AHWWLSPCJMALVHDDVSUGICL6RUCA,true
B082W3Z9YK,0,List(),B082W3Z9YK,5.0,"Quick delivery, fixed the issue!",1607225435363,Happy customer!,AHZIJGKEWRTAEOZ673G5B3SNXEGQ,true
B078W2BJY8,0,List(),B078W2BJY8,5.0,"I wasn't sure whether these were worth it or not, given the cost compared to the original branded filters.I can happily report that these are a great value and work every bit as good as the original. If you are on the fence worrying whether these are worth it- I can assure you they are.",1534104184306,Amazing value,AFGUPTDFAWOHHL4LZDV27ERDNOYQ,true
B08C9LPCQV,0,List(),B08C9LPCQV,5.0,Easy to install got the product expected to receive,1620176603754,Dryer parts,AELFJFAXQERUSMTXJQ6SYFFRDWMA,true


Metadata Dataset


main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
Industrial & Scientific,"ROVSUN Ice Maker Machine Countertop, Make 44lbs Ice in 24 Hours, Compact & Portable Ice Maker with Ice Basket for Home, Office, Kitchen, Bar (Silver)",3.7,61,"List(【Quick Ice Making】This countertop ice machine creates crystal & bullet shaped ice cubes; 44lbs of ice ready in 24 hours, 12 cubes made per cycle within 10 mins; you can perfectly use it for drinks, wine, smoothies, food, 【Portable Design】The weight of this ice maker is only 23.3lbs, and the small size (10.63 x14.37 x 12.87)"" makes it portable. It's compact feature is perfect for home, office, apartments, dormitories, RVs and more, it can be placed on countertop or tabletop, plug it anywhere you like, 【Simple Operation】Adding the water tank with purified water; Power on machine and press ""on/off"" button to start ice making process; After 8-12 minutes, ice cube will fall off into the ice basket automatically; Take it out and make you cool, 【Full Monitoring】 Designed with compressor cooling system, operates at low noise and will not disturb your normal life; See-through window on top allows you to easily view the progress and check ice level, 【1 Year Warranty】We do cover 1 year warranty on this ice maker, any questions about it, please don't hesitate to contact us at any time for getting a satisfied service. Worry free purchase, so get this ice machine home today)",List(),null,"List(Map(thumb -> https://m.media-amazon.com/images/I/31idkuA3KlL._SX38_SY50_CR,0,0,38,50_.jpg, large -> https://m.media-amazon.com/images/I/31idkuA3KlL.jpg, variant -> MAIN, hi_res -> https://m.media-amazon.com/images/I/61zNIJh6ZCL._SL1500_.jpg), Map(thumb -> https://m.media-amazon.com/images/I/41-MiODg9xL._SX38_SY50_CR,0,0,38,50_.jpg, large -> https://m.media-amazon.com/images/I/41-MiODg9xL.jpg, variant -> PT01, hi_res -> https://m.media-amazon.com/images/I/71i3VazBG1L._SL1500_.jpg), Map(thumb -> https://m.media-amazon.com/images/I/41wDXfH8m8L._SX38_SY50_CR,0,0,38,50_.jpg, large -> https://m.media-amazon.com/images/I/41wDXfH8m8L.jpg, variant -> PT02, hi_res -> https://m.media-amazon.com/images/I/71INhlaKSGL._SL1500_.jpg), Map(thumb -> https://m.media-amazon.com/images/I/41U-3N4Al4L._SX38_SY50_CR,0,0,38,50_.jpg, large -> https://m.media-amazon.com/images/I/41U-3N4Al4L.jpg, variant -> PT03, hi_res -> https://m.media-amazon.com/images/I/71IjZxwpPgL._SL1500_.jpg), Map(thumb -> https://m.media-amazon.com/images/I/51ZMIaw5j+L._SX38_SY50_CR,0,0,38,50_.jpg, large -> https://m.media-amazon.com/images/I/51ZMIaw5j+L.jpg, variant -> PT04, hi_res -> https://m.media-amazon.com/images/I/8154EOa6mML._SL1500_.jpg), Map(thumb -> https://m.media-amazon.com/images/I/41rxwKw1FqL._SX38_SY50_CR,0,0,38,50_.jpg, large -> https://m.media-amazon.com/images/I/41rxwKw1FqL.jpg, variant -> PT05, hi_res -> https://m.media-amazon.com/images/I/71P3bDmPIXL._SL1500_.jpg), Map(thumb -> https://m.media-amazon.com/images/I/5146wcOEdfL._SX38_SY50_CR,0,0,38,50_.jpg, large -> https://m.media-amazon.com/images/I/5146wcOEdfL.jpg, variant -> PT06, hi_res -> https://m.media-amazon.com/images/I/71ST326RerL._SL1500_.jpg), Map(thumb -> https://m.media-amazon.com/images/I/61-9E6JF51L._SX38_SY50_CR,0,0,38,50_.jpg, large -> https://m.media-amazon.com/images/I/61-9E6JF51L.jpg, variant -> PT07, hi_res -> https://m.media-amazon.com/images/I/81I6c2yZStL._SL1500_.jpg), Map(thumb -> https://m.media-amazon.com/images/I/51+Ddh3KfhL._SX38_SY50_CR,0,0,38,50_.jpg, large -> https://m.media-amazon.com/images/I/51+Ddh3KfhL.jpg, variant -> PT08, hi_res -> https://m.media-amazon.com/images/I/71Dap4gcywL._SL1500_.jpg))","List(Map(title -> Our Point of View on the Euhomy Ice Maker Machine, url -> https://www.amazon.com/vdp/04e6baae04404579891e175567cb7b9d?ref=dp_vse_rvc_0, user_id -> /shop/influencer-20a38664), Map(title -> Frigidaire Ice Maker, Stainless (5 Stars) , url -> https://www.amazon.c

In [0]:
print("=" * 70)
print("REVIEWS DATASET SCHEMA")
print("=" * 70)

reviews_df.printSchema()

print("\n")

print("=" * 70)
print("METADATA DATASET SCHEMA")
print("=" * 70)

meta_df.printSchema()

REVIEWS DATASET SCHEMA
root
 |-- asin: string (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- images: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- attachment_type: string (nullable = true)
 |    |    |-- large_image_url: string (nullable = true)
 |    |    |-- medium_image_url: string (nullable = true)
 |    |    |-- small_image_url: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- title: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- verified_purchase: boolean (nullable = true)



METADATA DATASET SCHEMA
root
 |-- main_category: string (nullable = true)
 |-- title: string (nullable = true)
 |-- average_rating: double (nullable = true)
 |-- rating_number: long (nullable = true)
 |-- features: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- des

In [0]:
reviews_dictionary = spark.createDataFrame(
    reviews_df.dtypes,
    ["Column Name", "Data Type"]
)

meta_dictionary = spark.createDataFrame(
    meta_df.dtypes,
    ["Column Name", "Data Type"]
)

print("Reviews Data Dictionary")
display(reviews_dictionary)

print("Metadata Data Dictionary")
display(meta_dictionary)

Reviews Data Dictionary


Column Name,Data Type
asin,string
helpful_vote,bigint
images,array>
parent_asin,string
rating,double
text,string
timestamp,bigint
title,string
user_id,string
verified_purchase,boolean


Metadata Data Dictionary


Column Name,Data Type
main_category,string
title,string
average_rating,double
rating_number,bigint
features,array
description,array
price,double
images,array>
videos,array>
store,string


In [0]:
import pyspark.sql.functions as F

In [0]:
# ============================================================
# Primary Key Validation - Metadata
# ============================================================

duplicate_parent_asin = (
    meta_df
    .groupBy("parent_asin")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_count = duplicate_parent_asin.count()

print(f"Duplicate parent_asin values : {duplicate_count}")

Duplicate parent_asin values : 0


In [0]:
# ============================================================
# Foreign Key Validation - Reviews
# ============================================================

missing_parent_asin = reviews_df.filter(
    F.col("parent_asin").isNull()
).count()

print(f"Missing parent_asin values : {missing_parent_asin}")

Missing parent_asin values : 0


In [0]:
# ============================================================
# Referential Integrity Check
# ============================================================

orphan_reviews = (
    reviews_df.alias("r")
    .join(
        meta_df.select("parent_asin").alias("m"),
        on="parent_asin",
        how="left_anti"
    )
)

print(f"Orphan Reviews : {orphan_reviews.count()}")

Orphan Reviews : 0


In [0]:
# ============================================================
# Reviews per Product
# ============================================================

reviews_per_product = (
    reviews_df
    .groupBy("parent_asin")
    .agg(F.count("*").alias("review_count"))
)

display(
    reviews_per_product
    .orderBy(F.desc("review_count"))
    .limit(10)
)

parent_asin,review_count
B0B3DB5HTC,12027
B07RNJY499,11620
B07WTXWC32,8519
B08YBGCNHP,7330
B000AST3AK,7030
B01KJ2FVFW,6140
B00UXG4WR8,5896
B000DLB2FI,5736
B081KSD3BK,5668
B09YC8YCV6,5401


In [0]:
# ============================================================
# Merge Reviews & Metadata
# ============================================================

merged_df = (
    reviews_df.alias("r")
    .join(
        meta_df.alias("m"),
        on="parent_asin",
        how="inner"
    )
    .select(
        "parent_asin",

        # -------- Reviews --------
        F.col("r.asin").alias("asin"),
        F.col("r.user_id").alias("user_id"),
        F.col("r.rating").alias("review_rating"),
        F.col("r.title").alias("review_title"),
        F.col("r.text").alias("review_text"),
        F.col("r.timestamp").alias("timestamp"),
        F.col("r.helpful_vote").alias("helpful_vote"),
        F.col("r.verified_purchase").alias("verified_purchase"),
        F.col("r.images").alias("review_images"),

        # -------- Metadata --------
        F.col("m.title").alias("product_title"),
        F.col("m.average_rating"),
        F.col("m.rating_number"),
        F.col("m.price"),
        F.col("m.store"),
        F.col("m.main_category"),
        F.col("m.categories"),
        F.col("m.features"),
        F.col("m.description"),
        F.col("m.details"),
        F.col("m.images").alias("product_images"),
        F.col("m.videos"),
        F.col("m.bought_together")
    )
)

In [0]:
# ============================================================
# Merge Success Rate
# ============================================================

reviews_count = reviews_df.count()
merged_count = merged_df.count()

merge_success = (merged_count / reviews_count) * 100

print(f"Review Records : {reviews_count:,}")
print(f"Merged Records : {merged_count:,}")
print(f"Merge Success  : {merge_success:.2f}%")

Review Records : 2,128,605
Merged Records : 2,128,605
Merge Success  : 100.00%


In [0]:
# ============================================================
# Dataset Overview
# ============================================================

print("=" * 60)
print("MERGED DATASET OVERVIEW")
print("=" * 60)

print(f"Rows    : {merged_df.count():,}")
print(f"Columns : {len(merged_df.columns)}")

display(merged_df.limit(5))

MERGED DATASET OVERVIEW
Rows    : 2,128,605
Columns : 23


parent_asin,asin,user_id,review_rating,review_title,review_text,timestamp,helpful_vote,verified_purchase,review_images,product_title,average_rating,rating_number,price,store,main_category,categories,features,description,details,product_images,videos,bought_together
B09Z2CBDDP,B09Z2DQ5VH,AFTQN4MTJHHVFM4FEXY5KT76OY4A,5.0,Nice organizer!,"We go through eggs like crazy, so I am loving this egg storage system. It saves a lot of space! I especially like that I can see exactly how many eggs we have on hand!!! The clear plastic is thick than I expected and well made. I also like that other items in the fridge can be set on top and not worry about them getting crushed. I expect this egg storage system to last us a long time and we've very pleased with it.",1668725686752,0,false,List(),"Egg Storage Container, Realife Automatic Rolling Egg Organizer Clear Plastic Holder for Refrigerator, 1 Layer",4.3,79,12.99,realife,Tools & Home Improvement,"List(Appliances, Parts & Accessories, Refrigerator Parts & Accessories, Egg Trays)","List([Auto Rolling Egg Storage]: The egg storage organizer for refrigerator adopts 7° slope design which makes eggs roll down automatically to the front place, easy to take out the egg from the holder without opening the lid., [ Stackable Layer Design]: Our egg container is designed with stackable layers which fixed with a card slot. Each layer can store 18 eggs keeping the refrigerator in order, no mess anymore. Semi-enclosed design can provide convenience for your life as well as good ventilation., [Transparent Egg Container ]: The automatic rolling egg organizer with lid is clearly visible adopting a transparent body which is easy to see and convenient for timely replenishment. The detachable cover is designed to load eggs conveniently. Storage grooves of the lid top can hold up to extra 9 eggs., [Food-Grade Plastic]: This egg storage container uses premium PET plastic, BPA free, durable and easy to clean. Low temperature resistant, suitable to keep in the fridge., [ Size for Wide Application ]:11""L×9""W×2.8""H one layer egg storage holder is not only perfect for refrigerator, but also for kitchen, restaurant, cabinets, table, countertop and racks.)",List(),"Map(Package Dimensions -> 11.97 x 9.57 x 3.27 inches, Pattern -> Single layer, Number of Pieces -> 1, Best Sellers Rank -> {""Tools & Home Improvement"": 157091, ""Refrigerator Egg Trays"": 216}, Size -> S, Batteries Required? -> No, Material -> Polypropylene, Plastic, Shape -> Rectangular, Manufacturer -> realife, Item Weight -> 1 pounds, Date First Available -> April 28, 2022, Room Type -> Kitchen, Handle Material -> Plastic, Brand -> Realife, Country of Origin -> China, Color -> Transparent, Batteries Included? -> No)","List(Map(thumb -> https://m.media-amazon.com/images/I/419J2mecTFL._AC_US75_.jpg, large -> https://m.media-amazon.com/images/I/419J2mecTFL._AC_.jpg, variant -> MAIN, hi_res -> https://m.media-amazon.com/images/I/71fa74TCNpL._AC_SL1500_.jpg), Map(thumb -> https://m.media-amazon.com/images/I/41cXAzPkPhL._AC_US75_.jpg, large -> https://m.media-amazon.com/images/I/41cXAzPkPhL._AC_.jpg, variant -> PT01, hi_res -> https://m.media-amazon.com/images/I/71Qgd+EEdVL._AC_SL1500_.jpg), Map(thumb -> https://m.media-amazon.com/images/I/41xzKnbOCGL._AC_US75_.jpg, large -> https://m.media-amazon.com/images/I/41xzKnbOCGL._AC_.jpg, variant -> PT02, hi_res -> https://m.media-amazon.com/images/I/71xWYH-S6SL._AC_SL1500_.jpg), Map(thumb -> https://m.media-amazon.com/images/I/31286KypXzL._AC_US75_.jpg, large -> https://m.media-amazon.com/images/I/31286KypXzL._AC_.jpg, variant -> PT03, hi_res -> https://m.media-amazon.com/images/I/61KQhWPfsZL._AC_SL1500_.jpg), Map(thumb -> https://m.media-amazon.com/images/I/41eUiwBJTSL._AC_US75_.jpg, large -> https://m.media-amazon.com/images/I/41eUiwBJTSL._AC_.jpg, variant -> PT04, hi_res -> https://m.media-amazon.com/images/I/71OSxGYZnHL._AC_SL1500_.jpg), Map(thumb -> https://m.media-amazon.com/images/I/41hatuFxCLL._AC_US75_.jpg

In [0]:
# ============================================================
# Merged Dataset Schema
# ============================================================

merged_df.printSchema()

root
 |-- parent_asin: string (nullable = true)
 |-- asin: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- review_rating: double (nullable = true)
 |-- review_title: string (nullable = true)
 |-- review_text: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- verified_purchase: boolean (nullable = true)
 |-- review_images: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- attachment_type: string (nullable = true)
 |    |    |-- large_image_url: string (nullable = true)
 |    |    |-- medium_image_url: string (nullable = true)
 |    |    |-- small_image_url: string (nullable = true)
 |-- product_title: string (nullable = true)
 |-- average_rating: double (nullable = true)
 |-- rating_number: long (nullable = true)
 |-- price: double (nullable = true)
 |-- store: string (nullable = true)
 |-- main_category: string (nullable = true)
 |-- categories: array (nullable = true)

In [0]:
import builtins

In [0]:
# ============================================================
# Enhanced Column Profiling Report
# ============================================================

from pyspark.sql.types import (
    StringType,
    IntegerType,
    LongType,
    DoubleType,
    FloatType,
    BooleanType,
    DateType,
    TimestampType,
)

total_rows = merged_df.count()

profiling = []

for field in merged_df.schema.fields:

    column_name = field.name
    datatype = field.dataType.simpleString()

    non_null_count = merged_df.filter(F.col(column_name).isNotNull()).count()

    null_count = total_rows - non_null_count

    null_percent = builtins.round((null_count / total_rows) * 100, 2)

    if isinstance(
        field.dataType,
        (
            StringType,
            IntegerType,
            LongType,
            DoubleType,
            FloatType,
            BooleanType,
            DateType,
            TimestampType,
        ),
    ):
        distinct_count = merged_df.select(column_name).distinct().count()
    else:
        distinct_count = None

    # -------------------------------
    # Suggested Role
    # -------------------------------

    if column_name == "parent_asin":
        role = "Primary/Foreign Key"

    elif column_name == "user_id":
        role = "User Identifier"

    elif column_name == "asin":
        role = "Product Variant"

    elif column_name == "review_rating":
        role = "Target Variable"

    elif column_name in [
        "review_text",
        "review_title",
        "description",
        "features"
    ]:
        role = "NLP Feature"

    elif column_name in [
        "price",
        "store",
        "average_rating",
        "rating_number",
        "main_category"
    ]:
        role = "Product Attribute"

    elif column_name == "timestamp":
        role = "Temporal Feature"

    else:
        role = "Auxiliary"

    profiling.append(

        (
            column_name,
            datatype,
            non_null_count,
            null_count,
            null_percent,
            distinct_count,
            role
        )
    )

profiling_df = spark.createDataFrame(

    profiling,

    [

        "Column",
        "Data Type",
        "Non Null",
        "Null",
        "Null %",
        "Distinct",
        "Suggested Role"

    ]

)

display(profiling_df)

Column,Data Type,Non Null,Null,Null %,Distinct,Suggested Role
parent_asin,string,2128605,0,0.0,94319,Primary/Foreign Key
asin,string,2128605,0,0.0,104237,Product Variant
user_id,string,2128605,0,0.0,1755732,User Identifier
review_rating,double,2128605,0,0.0,5,Target Variable
review_title,string,2128605,0,0.0,999079,NLP Feature
review_text,string,2128605,0,0.0,1840887,NLP Feature
timestamp,bigint,2128605,0,0.0,2104484,Temporal Feature
helpful_vote,bigint,2128605,0,0.0,533,Auxiliary
verified_purchase,boolean,2128605,0,0.0,2,Auxiliary
review_images,array>,2128605,0,0.0,null,Auxiliary


In [0]:
# ============================================================
# Products in Metadata with No Reviews
# ============================================================

products_without_reviews = (
    meta_df.alias("m")
    .join(
        reviews_df.select("parent_asin").distinct().alias("r"),
        on="parent_asin",
        how="left_anti"
    )
)

print("Products without reviews:", products_without_reviews.count())

Products without reviews: 8


In [0]:
# ============================================================
# Timestamp Conversion
# ============================================================

merged_df = (

    merged_df

    .withColumn(

        "review_timestamp",

        F.to_timestamp(
            F.from_unixtime(F.col("timestamp") / 1000)
        )

    )

    .drop("timestamp")

)

In [0]:
# ============================================================
# Temporal Features
# ============================================================

merged_df = (

    merged_df

    .withColumn("year", F.year("review_timestamp"))
    .withColumn("month", F.month("review_timestamp"))
    .withColumn("day", F.dayofmonth("review_timestamp"))
    .withColumn("quarter", F.quarter("review_timestamp"))
    .withColumn("week", F.weekofyear("review_timestamp"))

)

In [0]:
merged_df = merged_df.drop("bought_together")

In [0]:
# ============================================================
# Utility Function : Missing Value Report
# ============================================================

import builtins

def generate_missing_report(df):

    total_rows = df.count()

    # Single scan to calculate missing values
    missing_expr = [
        F.sum(
            F.when(F.col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in df.columns
    ]

    missing_counts = (
        df.select(*missing_expr)
          .first()
          .asDict()
    )

    report = []

    for column in df.columns:

        missing = missing_counts[column]

        missing_pct = builtins.round(
            (missing / total_rows) * 100,
            2
        )

        report.append(
            (
                column,
                missing,
                missing_pct
            )
        )

    report_df = spark.createDataFrame(
        report,
        [
            "Column",
            "Missing Values",
            "Missing %"
        ]
    )

    return report_df.orderBy(F.desc("Missing Values"))

In [0]:
missing_df = generate_missing_report(merged_df)

display(missing_df)

Column,Missing Values,Missing %
price,528346,24.82
main_category,28494,1.34
store,6064,0.28
parent_asin,0,0.0
asin,0,0.0
user_id,0,0.0
review_rating,0,0.0
review_title,0,0.0
review_text,0,0.0
helpful_vote,0,0.0


In [0]:
# ============================================================
# Utility Function : Duplicate Analysis
# ============================================================

import builtins

def duplicate_summary(df, subset):

    total_records = df.count()

    unique_records = df.dropDuplicates(subset).count()

    duplicate_records = total_records - unique_records

    duplicate_percentage = builtins.round(
        (duplicate_records / total_records) * 100,
        2
    )

    return spark.createDataFrame(

        [

            (

                total_records,

                unique_records,

                duplicate_records,

                duplicate_percentage

            )

        ],

        [

            "Total Records",

            "Unique Records",

            "Duplicate Records",

            "Duplicate %"

        ]

    )

In [0]:
display(

    duplicate_summary(

        merged_df,

        [

            "user_id",
            "parent_asin",
            "review_text",
            "review_timestamp"

        ]

    )

)

Total Records,Unique Records,Duplicate Records,Duplicate %
2128605,2105948,22657,1.06


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
# ============================================================
# Data Quality Check : Review Rating
# ============================================================

invalid_ratings = merged_df.filter(
    (~F.col("review_rating").between(1, 5))
)

print(f"Invalid Ratings : {invalid_ratings.count()}")

display(invalid_ratings.limit(10))

Invalid Ratings : 0


parent_asin,asin,user_id,review_rating,review_title,review_text,helpful_vote,verified_purchase,review_images,product_title,average_rating,rating_number,price,store,main_category,categories,features,description,details,product_images,videos,review_timestamp,year,month,day,quarter,week


In [0]:
# ============================================================
# Data Quality Check : Price
# ============================================================

negative_price = merged_df.filter(
    F.col("price") < 0
)

print(f"Negative Prices : {negative_price.count()}")

display(negative_price.limit(10))

Negative Prices : 0


parent_asin,asin,user_id,review_rating,review_title,review_text,helpful_vote,verified_purchase,review_images,product_title,average_rating,rating_number,price,store,main_category,categories,features,description,details,product_images,videos,review_timestamp,year,month,day,quarter,week


In [0]:
# ============================================================
# Data Quality Check : Review Timestamp
# ============================================================

future_reviews = merged_df.filter(
    F.col("review_timestamp") > F.current_timestamp()
)

print(f"Future Reviews : {future_reviews.count()}")

display(future_reviews.limit(10))

Future Reviews : 0


parent_asin,asin,user_id,review_rating,review_title,review_text,helpful_vote,verified_purchase,review_images,product_title,average_rating,rating_number,price,store,main_category,categories,features,description,details,product_images,videos,review_timestamp,year,month,day,quarter,week


In [0]:
# ============================================================
# Data Quality Dashboard
# ============================================================

quality_metrics = [

    ("Total Reviews", merged_df.count()),

    ("Unique Products",
     merged_df.select("parent_asin").distinct().count()),

    ("Unique Users",
     merged_df.select("user_id").distinct().count()),

    ("Missing Prices",
     merged_df.filter(F.col("price").isNull()).count()),

    ("Invalid Ratings",
     invalid_ratings.count()),

    ("Negative Prices",
     negative_price.count()),

    ("Future Reviews",
     future_reviews.count())

]

quality_dashboard = spark.createDataFrame(
    quality_metrics,
    ["Metric", "Value"]
)

display(quality_dashboard)

Metric,Value
Total Reviews,2128605
Unique Products,94319
Unique Users,1755732
Missing Prices,528346
Invalid Ratings,0
Negative Prices,0
Future Reviews,0


In [0]:
# ============================================================
# Save Cleaned Dataset
# ============================================================

CLEAN_DATASET_PATH = f"Volumes/project/default/data/clean/cleaned_merged_dataset"

(
    merged_df.write
    .mode("overwrite")
    .parquet(CLEAN_DATASET_PATH)
)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.
